# Day 4: Joining and Merging DataFrames

## Learning Objectives
By the end of this notebook, you will be able to:
- Understand why we need to join data from multiple sources
- Use merge() to combine DataFrames
- Understand different types of joins (inner, left, right, outer)
- Join on single and multiple columns
- Handle duplicate keys
- Use concat() to stack DataFrames
- Build integrated datasets from multiple files

---

## 1. Why Join Data?

In real business scenarios, data is rarely in one file. You typically have:
- **Sales transactions** (sales.csv): TransactionID, ProductID, CustomerID, Amount, Date
- **Product information** (products.csv): ProductID, ProductName, Category, Price
- **Customer data** (customers.csv): CustomerID, Name, Region, CustomerType

To analyze "Which customers in EMEA bought Premium products?", you need to **join** these tables together.

This is exactly like VLOOKUP in Excel, but more powerful!

### Demo 1.1: The Problem Without Joins

In [ ]:
# Demo: The Problem - Data in Separate Tables
# Real business data is often split across multiple files/tables
# Sales has IDs, but we need names from other tables

import pandas as pd

# Sales transactions - only has IDs, not human-readable names
sales = pd.DataFrame({
    'TransactionID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004'],
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-001', 'PROD-003'],
    'Revenue': [1500, 2000, 1800, 2500]
})

# Product master - has ProductID linked to names and categories
products = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-003'],
    'ProductName': ['Suite A', 'Suite B', 'Analytics'],
    'Category': ['Premium', 'Standard', 'Premium']
})

print("Sales (only IDs):")
print(sales)
print("\nProducts (has names):")
print(products)
print("\nProblem: Sales doesn't have product names! We need to JOIN.")

### Demo 1.2: The Solution - merge()

In [ ]:
# Demo: The Solution - merge()
# merge() joins two DataFrames on a common column (like VLOOKUP on steroids!)
# Syntax: df1.merge(df2, on='common_column')

# Join sales with products using ProductID as the matching key
sales_with_names = sales.merge(products, on='ProductID')

print("After joining:")
print(sales_with_names)
print("\nNow we have product names AND revenue in one table!")

---
## 2. Inner Join (Default)

**Inner join** keeps only rows where the key exists in **BOTH** DataFrames.

Think: "Show me only complete matches."

### Demo 2.1: Inner Join Behavior

In [ ]:
# Demo: Inner Join Behavior
# Inner join (default) keeps only rows where key exists in BOTH tables
# Rows without a match in BOTH tables are dropped

# Sales includes PROD-999 which doesn't exist in products!
sales = pd.DataFrame({
    'TransactionID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004'],
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-001', 'PROD-999'],  # PROD-999 doesn't exist!
    'Revenue': [1500, 2000, 1800, 2500]
})

products = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-003'],  # No PROD-999
    'ProductName': ['Suite A', 'Suite B', 'Analytics'],
    'Category': ['Premium', 'Standard', 'Premium']
})

# Inner join (default behavior with how='inner')
result = sales.merge(products, on='ProductID', how='inner')

print("Original sales rows: 4")
print("After inner join: ", len(result), "rows")
print("\nResult:")
print(result)
print("\nNote: TXN-004 (PROD-999) was dropped because PROD-999 doesn't exist in products!")

### Exercise 1: Inner Join Practice

Given customer and order data:
1. Join orders with customers on CustomerID
2. How many rows in the result?
3. Which customer has the highest total order value?

In [ ]:
# Exercise 1: Inner Join Practice
# Task: Join orders with customers and analyze

# Order data
orders = pd.DataFrame({
    'OrderID': ['ORD-001', 'ORD-002', 'ORD-003', 'ORD-004'],
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-A', 'CUST-C'],
    'Amount': [1500, 2000, 1800, 2500]
})

# Customer data
customers = pd.DataFrame({
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-C'],
    'CustomerName': ['Acme Corp', 'TechStart', 'Global Industries'],
    'Region': ['EMEA', 'AMER', 'APAC']
})

# 1. Join the data
# Use .merge() with customers on the common column 'CustomerID'
orders_with_customers = orders.merge(___)  # customers, on='CustomerID'

print("Joined data:")
print(orders_with_customers)

# 2. Count rows
print(f"\nRows in result: {len(orders_with_customers)}")

# 3. Find customer with highest total using groupby and idxmax
customer_totals = orders_with_customers.groupby('CustomerName')['Amount'].sum()
top_customer = customer_totals.idxmax()
top_amount = customer_totals.max()

print(f"\nTop customer: {top_customer} with ${top_amount:,}")

---
## 3. Left Join

**Left join** keeps **ALL** rows from the left DataFrame, even if there's no match in the right DataFrame.

Missing values become NaN.

Think: "Keep all my sales, even if product info is missing."

### Demo 3.1: Left Join Example

In [ ]:
# Demo: Left Join
# Left join keeps ALL rows from the LEFT table, even if no match in right
# Missing data becomes NaN - useful for finding data quality issues
# Syntax: df1.merge(df2, on='key', how='left')

# Same data with PROD-999 that doesn't exist in products
sales = pd.DataFrame({
    'TransactionID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004'],
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-001', 'PROD-999'],
    'Revenue': [1500, 2000, 1800, 2500]
})

products = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-003'],
    'ProductName': ['Suite A', 'Suite B', 'Analytics'],
    'Category': ['Premium', 'Standard', 'Premium']
})

# Left join - keep ALL sales, even those without matching product
result = sales.merge(products, on='ProductID', how='left')

print("Sales rows: 4")
print("After LEFT join: ", len(result), "rows")
print("\nResult:")
print(result)
print("\nNote: TXN-004 is kept, but ProductName and Category are NaN (missing)")

### Demo 3.2: When to Use Left Join

In [ ]:
# Demo: Using Left Join for Data Quality Checks
# Left join + isna() helps find "orphan" records - data quality issues!

result = sales.merge(products, on='ProductID', how='left')

# Find rows where ProductName is NaN (no match found)
# These are transactions referencing non-existent products
missing_product_info = result[result['ProductName'].isna()]

print("Transactions with missing product data:")
print(missing_product_info)
print(f"\nData quality issue: {len(missing_product_info)} transactions reference non-existent products!")

### Exercise 2: Left Join for Data Quality

Use left join to find data quality issues:
1. Join orders with customers (left join)
2. Find orders without customer information
3. Calculate total revenue from "orphan" orders

In [ ]:
# Exercise 2: Left Join for Data Quality
# Task: Find orders that reference non-existent customers

# Data with an orphan order (CUST-Z doesn't exist)
orders = pd.DataFrame({
    'OrderID': ['ORD-001', 'ORD-002', 'ORD-003', 'ORD-004', 'ORD-005'],
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-A', 'CUST-C', 'CUST-Z'],  # CUST-Z doesn't exist!
    'Amount': [1500, 2000, 1800, 2500, 3000]
})

customers = pd.DataFrame({
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-C'],
    'CustomerName': ['Acme Corp', 'TechStart', 'Global Industries']
})

# 1. Left join to keep ALL orders
# Use how='left' to keep all orders even without customer match
orders_check = orders.merge(customers, on='CustomerID', how=___)  # 'left'

# 2. Find orphan orders where CustomerName is null
# Use .isna() to find NaN values
orphan_orders = orders_check[orders_check['CustomerName'].___()]  # isna()

print("Orphan orders (no customer):")
print(orphan_orders)

# 3. Calculate total revenue from orphan orders
orphan_revenue = orphan_orders['Amount'].sum()
print(f"\nRevenue from orphan orders: ${orphan_revenue:,}")
print("These orders need investigation!")

---
## 4. Right and Outer Joins

- **Right join**: Keep all rows from RIGHT DataFrame (opposite of left)
- **Outer join**: Keep all rows from BOTH DataFrames

### Demo 4.1: Right Join

In [ ]:
# Demo: Right Join
# Right join keeps ALL rows from the RIGHT table
# Opposite of left join - keeps all products even without sales
# Syntax: df1.merge(df2, on='key', how='right')

# Right join - keep all products, even those with no sales
result = sales.merge(products, on='ProductID', how='right')

print("Result (all products kept):")
print(result)
print("\nNote: PROD-003 (Analytics) has NaN for TransactionID and Revenue (no sales)")

### Demo 4.2: Outer Join

In [ ]:
# Demo: Outer Join
# Outer join keeps EVERYTHING from BOTH tables
# Most inclusive - no data lost, but most NaN values
# Syntax: df1.merge(df2, on='key', how='outer')

# Outer join - keep all sales AND all products
result = sales.merge(products, on='ProductID', how='outer')

print("Result (all sales AND all products):")
print(result)
print("\nNote: Includes PROD-999 (no product info) AND PROD-003 (no sales)")

### Demo 4.3: Indicator Column

In [ ]:
# Demo: Using Indicator to Track Merge Source
# indicator=True adds a '_merge' column showing where each row came from
# Very useful for data quality analysis!

# Outer join with indicator
result = sales.merge(products, on='ProductID', how='outer', indicator=True)

print("With indicator column:")
print(result)

# Count each merge type
print("\nValue counts for _merge:")
print(result['_merge'].value_counts())
print("\n- both: matched in both DataFrames")
print("- left_only: only in sales (orphan transactions)")
print("- right_only: only in products (no sales yet)")

---
## 5. Joining on Multiple Columns

Sometimes you need to match on more than one column.

### Demo 5.1: Multi-Column Join

In [ ]:
# Demo: Joining on Multiple Columns
# Sometimes you need to match on more than one column
# Syntax: df1.merge(df2, on=['col1', 'col2'])

# Example: Regional pricing - different prices per region
sales = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-001', 'PROD-002'],
    'Region': ['EMEA', 'AMER', 'EMEA'],
    'Units': [100, 150, 200]
})

# Pricing varies by product AND region
pricing = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-001', 'PROD-002', 'PROD-002'],
    'Region': ['EMEA', 'AMER', 'EMEA', 'AMER'],
    'Price': [199, 189, 99, 94]  # Different prices by region!
})

# Join on BOTH ProductID AND Region to get correct price
result = sales.merge(pricing, on=['ProductID', 'Region'])

print("Sales with regional pricing:")
print(result)

# Calculate revenue using the correct regional price
result['Revenue'] = result['Units'] * result['Price']
print("\nWith revenue calculated:")
print(result)

### Exercise 3: Complex Join

Join sales with regional targets:
1. Join on Region AND Quarter
2. Calculate achievement % (Revenue / Target * 100)
3. Find which region-quarter combinations exceeded target

In [ ]:
# Exercise 3: Multi-Column Join
# Task: Join sales with regional targets and calculate achievement

sales = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'AMER', 'AMER', 'APAC'],
    'Quarter': ['Q1', 'Q2', 'Q1', 'Q2', 'Q1'],
    'Revenue': [125000, 138000, 98000, 105000, 67000]
})

targets = pd.DataFrame({
    'Region': ['EMEA', 'EMEA', 'AMER', 'AMER', 'APAC', 'APAC'],
    'Quarter': ['Q1', 'Q2', 'Q1', 'Q2', 'Q1', 'Q2'],
    'Target': [120000, 130000, 100000, 100000, 70000, 75000]
})

# Join on BOTH Region AND Quarter
# Must match on both columns to get correct target
result = sales.merge(targets, on=[___, ___])  # 'Region', 'Quarter'

# Calculate achievement percentage
result['Achievement_%'] = (result['Revenue'] / result['Target'] * 100).round(1)

print("Performance vs Target:")
print(result)

# Find region-quarters that exceeded target (>= 100%)
exceeded = result[result['Achievement_%'] >= 100]
print(f"\nRegion-quarters that exceeded target: {len(exceeded)}")
print(exceeded[['Region', 'Quarter', 'Achievement_%']])

---
## 6. Concatenating DataFrames

concat() is for **stacking** DataFrames (adding rows or columns), not joining.

### Demo 6.1: Concatenate Rows (Stack)

In [ ]:
# Demo: Concatenate Rows (Stacking DataFrames)
# concat() stacks DataFrames vertically (adds rows)
# Use for combining same-structure data from different periods
# Syntax: pd.concat([df1, df2], ignore_index=True)

# Monthly sales files - same structure, different months
jan_sales = pd.DataFrame({
    'Product': ['Suite A', 'Suite B'],
    'Revenue': [15000, 12000],
    'Month': ['Jan', 'Jan']
})

feb_sales = pd.DataFrame({
    'Product': ['Suite A', 'Suite B'],
    'Revenue': [18000, 13500],
    'Month': ['Feb', 'Feb']
})

# Stack them vertically
# ignore_index=True creates new sequential index (0, 1, 2, 3...)
all_sales = pd.concat([jan_sales, feb_sales], ignore_index=True)

print("Combined sales:")
print(all_sales)

### Demo 6.2: When to Use concat vs merge

- **concat()**: Stack DataFrames with same structure (like combining monthly files)
- **merge()**: Join DataFrames by matching on keys (like adding product names to sales)

In [ ]:
# Demo: Concat vs Merge - When to Use Each
# concat(): Stack same-structure data (like combining monthly files)
# merge(): Join different tables by matching on keys

# Step 1: Combine monthly files with concat()
q1_sales = pd.concat([jan_sales, feb_sales], ignore_index=True)

# Step 2: Enrich with product info using merge()
products = pd.DataFrame({
    'Product': ['Suite A', 'Suite B', 'Suite C'],
    'Category': ['Premium', 'Standard', 'Premium']
})

q1_enriched = q1_sales.merge(products, on='Product')

print("Q1 sales with categories:")
print(q1_enriched)

---
## Challenge Exercise: Build Integrated Dataset

You have three files:
1. Transactions (TransactionID, CustomerID, ProductID, Amount, Date)
2. Customers (CustomerID, CustomerName, Region, Type)
3. Products (ProductID, ProductName, Category, Cost)

Tasks:
1. Join all three DataFrames
2. Calculate profit (Amount - Cost)
3. Find total profit by Region and Category
4. Which Region-Category combination is most profitable?

In [ ]:
# Challenge Exercise: Build Integrated Dataset
# Task: Join three tables and calculate profit by Region and Category

# Transaction data
transactions = pd.DataFrame({
    'TransactionID': ['TXN-001', 'TXN-002', 'TXN-003', 'TXN-004', 'TXN-005'],
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-A', 'CUST-C', 'CUST-B'],
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-001', 'PROD-003', 'PROD-002'],
    'Amount': [1500, 2000, 1800, 2500, 1900],
    'Date': ['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19']
})

# Customer data
customers = pd.DataFrame({
    'CustomerID': ['CUST-A', 'CUST-B', 'CUST-C'],
    'CustomerName': ['Acme Corp', 'TechStart', 'Global Ind'],
    'Region': ['EMEA', 'AMER', 'APAC'],
    'Type': ['Premium', 'Standard', 'Premium']
})

# Product data
products = pd.DataFrame({
    'ProductID': ['PROD-001', 'PROD-002', 'PROD-003'],
    'ProductName': ['Suite A', 'Suite B', 'Analytics'],
    'Category': ['Premium', 'Standard', 'Premium'],
    'Cost': [800, 1100, 1300]
})

# Step 1: Join transactions with customers
# Use merge() with on='CustomerID'
df = transactions.merge(customers, on=___)  # 'CustomerID'

# Step 2: Join with products
# Chain another merge() with on='ProductID'
df = df.merge(products, on=___)  # 'ProductID'

# Step 3: Calculate profit
df['Profit'] = df['Amount'] - df['Cost']

print("Integrated dataset:")
print(df)

# Step 4: Profit summary by Region and Category
profit_summary = df.groupby(['Region', 'Category'])['Profit'].sum().reset_index()

print("\nProfit by Region and Category:")
print(profit_summary)

# Find most profitable combination using idxmax()
best = profit_summary.loc[profit_summary['Profit'].idxmax()]
print(f"\nMost profitable: {best['Region']} - {best['Category']} (${best['Profit']:,})")

---
## Summary

**You've learned:**
- Why joining data is essential
- merge() for combining DataFrames
- Different join types:
  - Inner: Only matches
  - Left: All from left + matches from right
  - Right: All from right + matches from left
  - Outer: Everything from both
- Joining on single and multiple columns
- Using indicator to track merge results
- concat() for stacking DataFrames

**Key Takeaways:**
- Left join is most common in business analysis (keep all your data)
- Use indicator=True to find data quality issues
- Multi-column joins handle more complex relationships
- concat() for same-structure stacking, merge() for joining

**Next:** Data visualization to present your integrated analysis!